## Demonstration of text splitting techniques with Python
---

In this notebook, we explore several text splitting techniques, useful for NLP (natural language processing) tasks, summarization, or indexing for conversational agents.

We will cover:

- Recursive splitting with overlap (LangChain)

- Tag-based splitting (HTML/Markdown)

- Semantic splitting with an NLP model (spaCy)

In [ ]:
# Installing the dependencies
!pip install langchain --quiet
!pip install spacy --quiet
!python -m spacy download en_core_web_sm

In [ ]:
text = """
# The Importance of Text Splitting (Chunking)

Text splitting, or "chunking", is a fundamental technique in natural language processing (NLP) and information management. It consists of dividing a long text into smaller, more manageable pieces called "chunks". This step is often crucial before applying other processing, such as indexing for search engines, sentiment analysis, or feeding large language models (LLMs) in systems like Retrieval-Augmented Generation (RAG).

## Why split the text?

There are several reasons. First, many AI models have a limit on the size of the context they can process at once (the "context window"). Splitting the text helps respect these limits. Second, for tasks like information retrieval, analyzing shorter, focused segments can yield better results than analyzing an entire document in a single block. This makes it possible to identify relevant passages more precisely. Finally, it makes it easier to parallelize processing over large volumes of textual data.

## Different Splitting Approaches

There is no single "right" way to split a text. The choice of method depends heavily on the type of content and the final goal. Here are a few common approaches we can compare:

* **Recursive character splitting:** This method aims to create chunks of an approximately fixed size (in number of characters), with some overlap so as not to lose context at the boundaries. It tries to cut on logical separators (paragraphs `\n\n`, sentences `. `, words ` `) before cutting abruptly if necessary to respect the size. It is **simple but potentially abrupt**.

* **Markdown-structure-based splitting:** Ideal for content written in Markdown. This technique uses the inherent structure of the document (headings `#`, `##`, etc., paragraphs separated by blank lines, lists `*` or `-`, code blocks ``` ```, quotes `>`) to define the chunks. A Markdown parser can identify these elements, and each can become a chunk. This **respects the organizational logic** of the document intended by the author.

* **Semantic splitting:** This more advanced approach uses NLP models to identify natural discourse boundaries, such as the end of sentences or groups of sentences dealing with the same sub-topic. The goal is to create chunks that are *semantically coherent*, even if their size varies. This requires more complex tools (such as spaCy or embedding models) but can preserve meaning more effectively.

## Concrete Example and Comparison

Let's imagine applying these three techniques to this very Markdown document.
Recursive splitting could cut the middle of a paragraph if it exceeds the target size, trying first to cut between paragraphs (on `\n\n`) or sentences.
Markdown-structure-based splitting would probably create a distinct chunk for each heading (`#`, `##`), each paragraph (block of text separated by a blank line), and each list item (`*`).
Semantic splitting would try to group sentences that talk about the same concept (for example, the explanation of recursive splitting) to form a chunk, even if it spans several lines or Markdown elements.

## Conclusion

In conclusion, the choice of chunking technique is an important design step. You have to consider the nature of the text (structured in Markdown or not, long or short) and the use that will be made of the chunks (search, summarization, feeding an LLM). A comparative analysis on real examples like this one is often necessary to choose the method best suited to your specific needs.
"""

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Initialize the splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,      # Size of each segment
    chunk_overlap=200     # Overlap between the segments
)

# Example of use

segments = text_splitter.split_text(text)

for i, segment in enumerate(segments):
    print(f"Segment {i+1}:\n{segment}\n")

In [ ]:
# Required import from LangChain
from langchain.text_splitter import MarkdownHeaderTextSplitter

# Define the headers on which we want to split the text.
# Each tuple represents a heading level ('#' for h1, '##' for h2, etc.)
# and the name associated with that level in the chunk metadata.
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
]

# Initialize the splitter
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

# Split the text
md_header_splits = markdown_splitter.split_text(text)

# Display the resulting chunks (each chunk is a LangChain Document
# with 'page_content' content and 'metadata')
for i, split in enumerate(md_header_splits):
    print(f"--- Chunk {i+1} ---")
    print(f"Metadata: {split.metadata}")
    print(f"Content:\n{split.page_content}")
    print("-" * 20)


In [ ]:
import spacy

# Load the English model
nlp = spacy.load("en_core_web_sm")

def semantic_chunking(text, max_chunk_size=1500):
    doc = nlp(text)
    segments = []
    current_segment = []

    for sent in doc.sents:
        if len(" ".join(current_segment)) + len(sent.text) <= max_chunk_size:
            current_segment.append(sent.text)
        else:
            segments.append(" ".join(current_segment))
            current_segment = [sent.text]
    if current_segment:
        segments.append(" ".join(current_segment))
    return segments

segments = semantic_chunking(text)

for i, segment in enumerate(segments):
    print(f"Segment {i+1}:\n{segment}\n")